# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amanparganiha/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git","clone","--depth","1",REPO_URL,REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)
print(sorted(df.columns.tolist()))

(30000, 44)
['age_tier', 'age_tier_order', 'ai_sessions_90d', 'ai_traffic_pct', 'avg_position', 'char_count', 'char_count_tier', 'clicks_90d', 'clicks_last_30d', 'clicks_prev_30d', 'client_id', 'competition', 'competition_level', 'content_age_days', 'content_id', 'content_type', 'cpc', 'ctr', 'days_since_last_update', 'days_with_impressions', 'days_with_sessions', 'engaged_sessions_90d', 'engagement_rate', 'freshness_tier', 'impression_tier', 'impressions_90d', 'impressions_last_30d', 'impressions_prev_30d', 'main_intent', 'model_used', 'pageviews_90d', 'position_tier', 'provider_used', 'scroll_events_90d', 'scroll_rate', 'search_volume', 'sessions_90d', 'sessions_last_30d', 'sessions_prev_30d', 'trend_direction', 'trend_pct', 'users_90d', 'word_count', 'word_count_tier']


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is CTR / Engagement Opportunity Scoring. The task type is ranking (via supervised scoring), not plain classification.

The output is an ordered queue: given all pages that already have search visibility, rank them by how much click opportunity they are leaving on the table. A reviewer works down that list from the top. Because only the top of the list is ever acted on, the model is judged on ordering quality at small K, not on global accuracy.

I train it as a supervised problem with a binary target (under-capturing or not) and use the predicted probability as the ranking score. That keeps the model honest — it learns from a defined label rather than an arbitrary formula — while the deliverable stays a ranked queue.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
work = df[(df["impressions_prev_30d"] >= 100) &
          (df["impressions_last_30d"] >= 100)].copy()

work["ctr_prev"] = 100 * work["clicks_prev_30d"] / work["impressions_prev_30d"]
work["ctr_last"] = 100 * work["clicks_last_30d"] / work["impressions_last_30d"]

print("Working population:", len(work), "of", len(df), "rows")
print("Clients:", work["client_id"].nunique(),
      "| Content items:", work["content_id"].nunique())
print("\nCTR (%) by window:")
print(work[["ctr_prev","ctr_last"]].describe().round(3).to_string())

Working population: 15615 of 30000 rows
Clients: 29 | Content items: 15615

CTR (%) by window:
        ctr_prev   ctr_last
count  15615.000  15615.000
mean       0.262      0.330
std        0.371      0.440
min        0.000      0.000
25%        0.000      0.000
50%        0.143      0.198
75%        0.364      0.479
max        6.776      7.463


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

My target is a defined rule, not an observed outcome — there is no ground-truth
"this page underperformed" flag in the data.

The naive proxy, low CTR, is wrong: CTR is dominated by position. A page at rank 8
with 1% CTR is normal; a page in the top 3 with 1% CTR is not. Predicting raw CTR
would mostly rediscover ranking.

So my label is position-conditional and time-aware:

    expected_ctr(tier) = median ctr_prev within that position tier
    under_capturing    = ctr_last < expected_ctr(tier)

Expected CTR is learned on the prev-30 window. The label is read from the last-30
window. Features come only from prev-30, so nothing from the outcome window can
enter the model.

That separation matters. In notebook 02 a depth-2 tree hit Precision@50 = 1.000 by
splitting on trend_pct, because the label was derived from it. Here the equivalent
trap is any last-30 column, so I exclude them explicitly rather than relying on
having remembered.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
exp = work.groupby("position_tier")["ctr_prev"].median().rename("expected_ctr")
work = work.join(exp, on="position_tier")

work["ctr_gap_last"] = work["ctr_last"] - work["expected_ctr"]
work["under_capturing"] = (work["ctr_gap_last"] < 0).astype(int)

print(work.groupby("position_tier")[["ctr_prev","expected_ctr","ctr_last"]]
        .median().round(3).to_string())
print("\nLabel rate:", round(work["under_capturing"].mean(), 3))
print("\nLabel rate by position tier:")
print(work.groupby("position_tier")["under_capturing"].mean().round(3).to_string())

BANNED = ["ctr","clicks_90d","clicks_last_30d","ctr_last","ctr_gap_last",
          "expected_ctr","trend_direction","trend_pct","sessions_last_30d",
          "impressions_last_30d","pageviews_90d","users_90d"]
print("\nExcluded from features (outcome-window or label-derived):")
for c in BANNED: print("  -", c)

               ctr_prev  expected_ctr  ctr_last
position_tier                                  
deep              0.000         0.000     0.000
page_1            0.234         0.234     0.276
page_3_5          0.030         0.030     0.072
striking          0.138         0.138     0.204
top_3             0.254         0.254     0.515

Label rate: 0.432

Label rate by position tier:
position_tier
deep        0.000
page_1      0.446
page_3_5    0.457
striking    0.435
top_3       0.289

Excluded from features (outcome-window or label-derived):
  - ctr
  - clicks_90d
  - clicks_last_30d
  - ctr_last
  - ctr_gap_last
  - expected_ctr
  - trend_direction
  - trend_pct
  - sessions_last_30d
  - impressions_last_30d
  - pageviews_90d
  - users_90d


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Metric: Precision@50 on held-out clients, against a hand-rule baseline on the
identical split.

Precision@K because the deliverable is a review queue and a content team only works
the top of it. K=50 to stay comparable with the reference pipeline's benchmark.

"Good" means beating the hand rule by a margin that survives client-holdout
validation. In notebook 02 I found an in-sample comparison overstated a tree's
advantage: under GroupShuffleSplit on client id, the depth-2 tree scored 0.640
against the hand rule's 0.680 — worse on unseen clients. So the bar is the held-out
margin, not the in-sample one.

Secondary check: client coverage. A queue that fills 45 of its top 50 slots with one
client's pages is not usable, whatever its precision. With only 32 clients in this
dataset, a grouped split is lumpy and single-seed results should be read as
directional.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

y = work["under_capturing"].values
groups = work["client_id"].values

hand = (work["impressions_prev_30d"].rank(pct=True) *
        work["avg_position"].rank(pct=True)).values

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr, te = next(gss.split(work, y, groups=groups))

print("Train clients:", len(set(groups[tr])), "| Test clients:", len(set(groups[te])))
print("Test rows:", len(te), "| Test label rate:", round(y[te].mean(), 3))
print("\nHand-rule Precision@50 (held-out clients):",
      round(precision_at_k(hand[te], y[te], 50), 3))

top50 = np.argsort(-hand[te])[:50]
print("Distinct clients in top 50:", len(set(groups[te][top50])))

Train clients: 21 | Test clients: 8
Test rows: 3083 | Test label rate: 0.431

Hand-rule Precision@50 (held-out clients): 0.18
Distinct clients in top 50: 3


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one content item, for one client, over a 30-day window.

Not a query, not a day, not a client. The action attaches to a page: a reviewer opens
one page and rewrites its title or meta description. Modelling at query level would
produce output nobody can act on.

The starter CSV is already at this grain — 30,000 rows, 30,000 unique content_id,
32 clients. In the full warehouse this is dim_content joined to per-content
aggregates of fact_content_query_90d, which is why the framing prototyped here
carries over unchanged.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
cols = ["client_id","content_id","content_type","position_tier","avg_position",
        "impressions_prev_30d","ctr_prev","word_count","content_age_days",
        "days_since_last_update","expected_ctr","ctr_last","under_capturing"]
print("One row = one content item, one client, 30-day window\n")
display(work[cols].head(10))
print("\nGrain check — rows:", len(work),
      "| unique content_id:", work["content_id"].nunique())


One row = one content item, one client, 30-day window



,client_id,content_id,content_type,position_tier,avg_position,impressions_prev_30d,ctr_prev,word_count,content_age_days,days_since_last_update,expected_ctr,ctr_last,under_capturing
0,client_f369cb89fc,content_304f48230142,keyword article,striking,10.6,987,1.317123,3221.0,187,20,0.138090,0.346021,0
1,client_4e07408562,content_a1fb4e703a9e,keyword article,page_3_5,20.3,5915,0.016906,2481.0,445,25,0.029709,0.079968,0
2,client_7f2253d7e2,content_9aa793d4d895,keyword article,page_3_5,36.5,6089,0.049269,3515.0,141,20,0.029709,0.041982,0
3,client_19581e27de,content_331d6c4de07b,keyword article,page_1,6.2,4206,0.404184,NaN,463,22,0.233981,0.606729,0
4,client_3fdba35f04,content_d99b7a2d90ca,keyword article,page_3_5,44.0,6452,0.030998,2803.0,263,14,0.029709,0.237473,0
5,client_f369cb89fc,content_d4084a4bc775,keyword article,page_1,8.5,1009,0.099108,3080.0,147,20,0.233981,0.000000,1
7,client_19581e27de,content_a63219c6e95a,keyword article,page_3_5,21.2,632,0.000000,NaN,445,22,0.029709,0.157233,0
8,client_6208ef0f77,content_5e6c160719bc,keyword article,page_3_5,46.0,13828,0.057854,3807.0,90,20,0.029709,0.158006,0
9,client_19581e27de,content_c27558df2b0c,keyword article,page_1,4.9,356,0.000000,NaN,257,104,0.233981,0.000000,1
10,client_19581e27de,content_d8ee6cc6d642,keyword article,top_3,2.2,6441,1.816488,NaN,329,104,0.254453,1.461187,0



Grain check — rows: 15615 | unique content_id: 15615


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule would be "flag any page with CTR below 2%." Three reasons that fails.

The threshold is not constant. Expected CTR differs by position tier, and my
notebook 01 finding showed it differs sharply by content type too — at page_1,
comparison articles averaged 0.13% CTR against feedly articles' 3.35%, with only
~18% of comparison articles getting any clicks at all. One cutoff mislabels whole
content types.

The signal is an interaction. Whether low CTR is a problem depends on position AND
content type AND impression volume together. A hand rule adds these as independent
terms; a tree can learn the interaction.

The distribution is zero-inflated. Median CTR across the full dataset is 0.07% and
the 25th percentile is 0. Median-based rules collapse. Any workable rule needs an
impressions floor plus a distribution-aware comparison — at which point you are
already fitting a model.

What ML does not buy me: causality. This ranks pages by association with
under-capture in an anonymized snapshot. It is decision-support for prioritising
human review, not evidence about how Google ranks anything.

The action it supports: the top of the queue becomes a metadata review list — title
and description rewrites on pages that already have impressions, where a CTR change
converts existing visibility into clicks without needing a ranking change.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
tbl = work.groupby(["position_tier","content_type"]).agg(
    n=("under_capturing","size"),
    label_rate=("under_capturing","mean"),
    median_ctr_prev=("ctr_prev","median")).round(3)
print(tbl.to_string())
print("\nSpread in label_rate across cells:",
      round(tbl["label_rate"].max() - tbl["label_rate"].min(), 3))
print("A single global CTR threshold cannot serve cells with baselines this different.")

                                     n  label_rate  median_ctr_prev
position_tier content_type                                         
deep          feedly article         2       0.000            0.000
              keyword article      376       0.000            0.000
page_1        comparison article    43       0.907            0.000
              feedly article        81       0.506            0.299
              keyword article     6451       0.442            0.235
page_3_5      comparison article     8       0.750            0.000
              feedly article        11       0.545            0.231
              keyword article     4122       0.457            0.030
striking      comparison article    10       0.600            0.000
              feedly article        30       0.533            0.000
              keyword article     4128       0.433            0.139
top_3         feedly article         3       0.333            2.495
              keyword article      350       0.2

In [14]:
# Boundary check: where does the label definition break down?
zero_both = ((work["ctr_prev"] == 0) & (work["ctr_last"] == 0))
print("Rows with zero CTR in BOTH windows:", zero_both.sum(),
      f"({zero_both.mean():.1%})")
print("Their label rate:", round(work.loc[zero_both, "under_capturing"].mean(), 3))

print("\nBy tier — share of rows that are zero in both windows:")
print(work.groupby("position_tier").apply(
    lambda g: ((g["ctr_prev"] == 0) & (g["ctr_last"] == 0)).mean()
).round(3).to_string())

print("\nCells with n < 20 (too small to read):")
small = tbl[tbl["n"] < 20]
print(small.to_string() if len(small) else "none")

Rows with zero CTR in BOTH windows: 3256 (20.9%)
Their label rate: 0.908

By tier — share of rows that are zero in both windows:
position_tier
deep        0.788
page_1      0.095
page_3_5    0.327
striking    0.227
top_3       0.088

Cells with n < 20 (too small to read):
                                   n  label_rate  median_ctr_prev
position_tier content_type                                       
deep          feedly article       2       0.000            0.000
page_3_5      comparison article   8       0.750            0.000
              feedly article      11       0.545            0.231
striking      comparison article  10       0.600            0.000
top_3         feedly article       3       0.333            2.495


/tmp/ipykernel_2065/514933292.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(work.groupby("position_tier").apply(


Boundary check — where this label definition breaks.

The `deep` tier has a label rate of exactly 0.000 across 378 rows. That is not
evidence that deep pages capture clicks well; it is an artifact. Their expected CTR
is 0 and their actual CTR is 0, so `ctr_last < expected_ctr` is `0 < 0`, which is
false for every row. The definition cannot express "under-capturing" when both sides
are zero.

Two consequences I carry forward:

1. The working population needs a stricter floor than "100 impressions." Pages with
   no clicks in either window are not opportunities — there is nothing to recover.
   In ML-04 I will require non-zero clicks in the prev window, and report how many
   rows that removes.

2. Several content_type cells are too small to read: deep/feedly n=2, top_3/feedly
   n=3, page_3_5/comparison n=8. Their label rates (0.000, 0.333, 0.750) are noise.
   I will not treat per-cell rates below n=20 as signal.

The honest version of the section 5 claim is therefore narrower: the label rate
varies from 0.29 to 0.91 across cells with adequate sample size, which still defeats
a single global threshold — but the extreme values in the table above are artifacts
of the definition, not findings.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.